In [ ]:
#Ollama needs zstd installed to work with Google Colab notebooks
!sudo apt-get install zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# Start the command in a background process
process = subprocess.Popen(["ollama", "serve"])

# Give Ollama server a few seconds to start up
print("Waiting for Ollama server to start...")
time.sleep(10) # Adjust as needed

# The kernel can continue execution while the process runs in the background
print("The 'ollama serve' process is running in the background and should be ready.")

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
!pip install ollama

In [6]:
import ollama

In [7]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]



In [ ]:

ksb = fetch_website_contents("https://github.com/karan-singhh")
ksb

In [9]:
# Define our system prompt

system_prompt = """
You are Tyrion Lannister.

Read website content and write a very short summary in Tyrion's voice.

Rules:

* Stay in character.
* Be witty, sarcastic, and observant.
* Mock marketing fluff and corporate buzzwords.
* Use occasional references to Westeros, noble houses, schemes, dragons, or the Small Council.
* Ignore navigation, menus, footers, and cookie notices.
* Keep responses under 80 words.
* Use markdown.
* Never explain the joke.

"""

In [10]:
# Define our user prompt

user_prompt_prefix = """
Tyrion,

Summarize this website for the Small Council.

Be funny, snarky, and concise.

Website content:


"""

In [11]:
# this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [13]:
from openai import OpenAI

In [14]:
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)



In [15]:
# call the OpenAI API.

def summarize(url):
    website = fetch_website_contents(url)
    response = client.chat.completions.create(
        model='llama3.2:1b',
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [26]:
# summarize("https://github.com/karan-singhh")


In [21]:
from IPython.display import Markdown, display


In [19]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://github.com/karan-singhh")